<a href="https://colab.research.google.com/github/priya25-art/Machine-Learning/blob/main/Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Step 1: Import libraries
import pandas as pd
import numpy as np
import os
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Step 2: Load dataset (with automatic upload prompt if missing)
file_path = "Practice_data.xlsx"
if not os.path.exists(file_path):
    print("Practice_data.xlsx not found. Please upload it below:")
    uploaded = files.upload()

df = pd.read_excel(file_path)

# Step 3: Inspect dataset
print("Dataset Info:\n", df.info())
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicate Records:", df.duplicated().sum())

# Step 4: Handle missing values without using inplace=True to prevent FutureWarnings
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Step 5: Remove duplicates
df = df.drop_duplicates()

# Step 6: Separate features (X) and target (y)
X = df.drop(columns=["Final_Score", "Student_ID"])
y = df["Final_Score"]

# Step 7: Identify categorical and numeric columns
categorical_cols = ["Department", "Gender"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# Step 8: Preprocessing (scaling + encoding)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="first"), categorical_cols)
    ]
)

# Step 9: Build pipeline with Linear Regression
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Step 10: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 11: Train model
model.fit(X_train, y_train)

# Step 12: Evaluate model
y_pred = model.predict(X_test)
print("R² Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

# Step 13: Feature importance (coefficients)
# Extract coefficients after preprocessing
feature_names = numeric_cols + list(model.named_steps["preprocessor"].transformers_[1][1].get_feature_names_out(categorical_cols))
coefficients = model.named_steps["regressor"].coef_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
}).sort_values(by="Coefficient", ascending=False)

print("\nFeature Importance:\n", importance_df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Student_ID             28 non-null     int64  
 1   Department             28 non-null     object 
 2   Gender                 28 non-null     object 
 3   Attendance_Pct         27 non-null     float64
 4   Study_Hours_Per_Day    27 non-null     float64
 5   Internal_Marks         27 non-null     float64
 6   Assignments_Completed  27 non-null     float64
 7   Final_Score            28 non-null     int64  
dtypes: float64(4), int64(2), object(2)
memory usage: 1.9+ KB
Dataset Info:
 None

Missing Values:
 Student_ID               0
Department               0
Gender                   0
Attendance_Pct           1
Study_Hours_Per_Day      1
Internal_Marks           1
Assignments_Completed    1
Final_Score              0
dtype: int64

Duplicate Records: 3
R² Score: 0.892715607358701